# Part 2 · From Analysis to Online Dashboard with Streamlit

<br>

## Tutorial Overview

In Part 1, you explored climate data with pandas and Altair, and you built a storyline around South Korea’s warming trend: **Hook → Evidence → Meaning**. Now you will turn that storyline into an interactive online dashboard using **Streamlit** and deploy it for free on **Streamlit Community Cloud**.

By the end of this tutorial, your analysis will be a live URL that anyone can visit.

<br>

<table>
<tr><td width="140"><b>Duration</b></td><td>~1.5 hours</td></tr>
<tr><td><b>Level</b></td><td>Beginner to Intermediate</td></tr>
<tr><td><b>Prerequisites</b></td><td>Part 1 completion, basic Python</td></tr>
</table>

<br>

## Learning Objectives

By the end of this tutorial, you will be able to:

- Translate a **data story** (Hook → Evidence → Meaning) into a dashboard layout
- Build an interactive climate dashboard with **Streamlit**
- Connect **Altair charts** to user controls so viewers can explore the story themselves
- Generate a **dynamic narrative** that updates based on the selected data
- Deploy the dashboard to **Streamlit Community Cloud** for free

<br>

---

## Step 1 · Map Your Story to a Dashboard

Before writing any code, plan how Part 1’s storyline maps onto a dashboard structure.

<br>

### 1.1 Recall the Story Structure from Part 1

In Part 1, you built a narrative with three parts:

<table>
<tr><td width="120"><b>Hook</b></td><td>"South Korea’s temperature anomaly has risen sharply since the mid-20th century."</td></tr>
<tr><td><b>Evidence</b></td><td>Time series, warming stripes, decade bars, multi-country comparison</td></tr>
<tr><td><b>Meaning</b></td><td>Recent decades show accelerating warming; how close is this to the 1.5°C Paris target?</td></tr>
</table>

<br>

### 1.2 Dashboard Layout Plan

Translate that story into dashboard sections:

<table>
<tr><td width="220"><b>Story Element</b></td><td><b>Dashboard Component</b></td></tr>
<tr><td>Hook (key numbers)</td><td>Metric cards at the top: latest anomaly, hottest year, average</td></tr>
<tr><td>Evidence (charts)</td><td>Tabs: Time Series · Warming Stripes · Decades</td></tr>
<tr><td>Meaning (narrative)</td><td>Auto-generated summary text + Paris Agreement progress bar</td></tr>
<tr><td>Interactivity</td><td>Sidebar controls: country selector, year range slider, toggle options</td></tr>
</table>

<br>

### 1.3 Why Interactivity Strengthens the Story

A static chart tells one story. An interactive dashboard lets the audience:

- Compare countries and discover that warming patterns vary by region
- Adjust the time range and see how trends change over different periods
- Toggle overlays (trend line, moving average) and understand what the raw data hides

👉 Reflect: Look at the charts you exported in Part 1. Which ones would benefit most from user interaction?

<br>

---

## Step 2 · Set Up the Project

In this step, you will create the project structure needed for development and deployment.

<br>

### 2.1 Install Streamlit

In [ ]:
# Run this in your terminal (not inside Jupyter):
# pip install streamlit

import streamlit
print(f"Streamlit version: {streamlit.__version__}")

### 2.2 Create the Project Structure

```text
climate-dashboard/
├── app.py                                        ← main dashboard
├── requirements.txt                              ← dependencies for deployment
├── data/
│   └── ddbb_surface_temperature_countries.csv    ← dataset from Part 1
└── README.md
```

Create this structure now:

```bash
mkdir -p climate-dashboard/data
cd climate-dashboard
```

Copy your dataset from Part 1 into the `data/` folder.

<br>

### 2.3 Create `requirements.txt`

```text
streamlit
pandas
altair
numpy
scipy
```

### 2.4 Verify the Setup

```bash
streamlit hello
```

If a demo app opens in the browser, Streamlit is installed and working.

👉 Checkpoint: Make sure your project folder has the structure shown above before continuing.

<br>

---

## Step 3 · Build the Data Layer with Caching

Streamlit reruns the entire script every time a user interacts with a widget. Without caching, the CSV file reloads on every click, which slows the app.

<br>

### 3.1 How Streamlit’s Execution Model Works

```text
User changes a widget → entire script reruns top to bottom → UI updates
```

This means expensive operations like loading data need to be cached.

<br>

### 3.2 Load and Cache the Data

In [ ]:
data_layer_code = '''
import streamlit as st
import pandas as pd

@st.cache_data
def load_data():
    """Load and clean the Berkeley Earth dataset (runs only once)."""
    df = pd.read_csv("data/ddbb_surface_temperature_countries.csv")
    df = df.dropna(subset=["Anomaly"])
    return df

@st.cache_data
def get_annual_data(df, country):
    """Aggregate monthly records into annual averages for a country."""
    country_data = df[df["Country"] == country]
    annual = country_data.groupby("Years").agg({
        "Anomaly": "mean",
        "Temperature": "mean"
    }).reset_index()
    annual.columns = ["Year", "Anomaly", "Temperature"]
    return annual

@st.cache_data
def get_decade_data(annual_df):
    """Calculate decade averages from annual data."""
    df = annual_df.copy()
    df["Decade"] = (df["Year"] // 10) * 10
    return df.groupby("Decade")["Anomaly"].mean().reset_index()
'''

print(data_layer_code)

### 3.3 Why These Three Functions?

Each function corresponds to a layer of the Part 1 analysis pipeline:

<table>
<tr><td width="200"><code>load_data()</code></td><td>Same as the initial <code>pd.read_csv()</code> + <code>dropna()</code> from Part 1 Step 3</td></tr>
<tr><td><code>get_annual_data()</code></td><td>Same as the <code>groupby('Years')</code> aggregation from Part 1 Step 4</td></tr>
<tr><td><code>get_decade_data()</code></td><td>Same as the decade grouping from Part 1 Step 4</td></tr>
</table>

The `@st.cache_data` decorator stores the result so that the function only runs when its inputs change.

👉 Checkpoint: Match each function above to the corresponding cell in your Part 1 notebook.

<br>

---

## Step 4 · Create the Sidebar Controls

The sidebar is where users make choices. Their selections drive every chart and narrative block on the page.

<br>

### 4.1 Sidebar Code

In [ ]:
sidebar_code = '''
# Page Configuration
st.set_page_config(
    page_title="Climate Data Story",
    page_icon="\U0001f30d",
    layout="wide"
)

# Load Data
df = load_data()
countries = sorted(df["Country"].unique())

# Sidebar
st.sidebar.title("\U0001f30d Climate Explorer")
st.sidebar.markdown("Explore temperature trends by country and time period.")

selected_country = st.sidebar.selectbox(
    "Select Country",
    countries,
    index=countries.index("South Korea") if "South Korea" in countries else 0
)

year_range = st.sidebar.slider(
    "Year Range",
    min_value=1900,
    max_value=2020,
    value=(1900, 2020)
)

st.sidebar.markdown("---")
st.sidebar.subheader("Chart Options")
show_trend = st.sidebar.checkbox("Show Trend Line", value=True)
show_ma = st.sidebar.checkbox("Show Moving Average", value=True)
'''

print(sidebar_code)

### 4.2 Design Choices

- **`selectbox`** for country: users focus on one country’s story at a time
- **`slider`** for year range: users can zoom into specific periods (e.g., 1960–2020)
- **`checkbox`** for overlays: users control the visual complexity

These controls map directly to the analytical decisions you made manually in Part 1 (choosing a country, filtering by date, adding trend lines).

👉 Try: Think about which default values tell the strongest initial story.

<br>

---

## Step 5 · Build the Hook: Metric Cards

The first thing the user sees should be the **Hook** — the key numbers that immediately signal the story.

<br>

### 5.1 Compute and Display Metrics

In [ ]:
metrics_code = '''
# Prepare Data
annual = get_annual_data(df, selected_country)
annual_filtered = annual[
    (annual["Year"] >= year_range[0]) &
    (annual["Year"] <= year_range[1])
]

# Title
st.title(f"Climate Story: {selected_country}")

# Hook: Key Metrics
col1, col2, col3, col4 = st.columns(4)

latest = annual_filtered.iloc[-1]
avg = annual_filtered["Anomaly"].mean()
hottest = annual_filtered.loc[annual_filtered["Anomaly"].idxmax()]
coldest = annual_filtered.loc[annual_filtered["Anomaly"].idxmin()]

with col1:
    st.metric("Latest Anomaly", f"{latest[chr(39)+chr(39)]}")
    # st.metric("Latest Anomaly", f"{latest_val:.2f}\u00b0C")
with col2:
    st.metric("Period Average", f"{avg:.2f}\u00b0C")
with col3:
    st.metric(f"Hottest ({int(hottest[chr(39)+chr(39)])})", f"{hottest_val:.2f}\u00b0C")
with col4:
    st.metric(f"Coldest ({int(coldest_yr)})", f"{coldest_val:.2f}\u00b0C")
'''

# Note: The actual working code is in the complete app.py (Step 9)
print("See the complete app.py in Step 9 for the working metrics code.")

### 5.2 Why Metrics First?

In Part 1 Step 8, you learned the **Hook → Evidence → Meaning** structure. The metric cards serve as the Hook: they immediately tell the user that the latest temperature anomaly is positive and that there is a significant gap between the coldest and hottest years.

This is exactly like a news headline — it grabs attention before the evidence is presented.

👉 Observe: The metric values change automatically when the user selects a different country or year range. The hook adapts to whatever story the data tells.

<br>

---

## Step 6 · Build the Evidence: Tabbed Charts

Tabs organize multiple pieces of evidence without overwhelming the viewer. Each tab presents a different angle on the same story.

<br>

### 6.1 Tab Structure

```python
st.markdown("---")
tab1, tab2, tab3 = st.tabs(["\U0001f4c8 Time Series", "\U0001f321\ufe0f Warming Stripes", "\U0001f4ca Decades"])
```

<br>

### 6.2 Tab 1: Time Series (from Part 1 Steps 6.3–6.4)

This is the core evidence chart. It corresponds to the trend chart you built in Part 1, enhanced with optional overlays.

In [ ]:
timeseries_code = '''
with tab1:
    plot_data = annual_filtered.copy()
    plot_data["MA_10"] = plot_data["Anomaly"].rolling(10, center=True).mean()

    base = alt.Chart(plot_data).encode(x=alt.X("Year:Q", title="Year"))

    # Data points
    points = base.mark_circle(size=40, opacity=0.6, color="steelblue").encode(
        y=alt.Y("Anomaly:Q", title="Temperature Anomaly (\u00b0C)"),
        tooltip=["Year:Q", alt.Tooltip("Anomaly:Q", format=".2f")]
    )
    layers = [points]

    # Optional: Moving average (red line)
    if show_ma:
        ma = base.mark_line(color="#e74c3c", strokeWidth=2.5).encode(y="MA_10:Q")
        layers.append(ma)

    # Optional: Trend line (green dashed)
    if show_trend:
        trend = base.transform_regression("Year", "Anomaly").mark_line(
            color="#27ae60", strokeDash=[5, 5], strokeWidth=2
        ).encode(y="Anomaly:Q")
        layers.append(trend)

    # Reference: baseline at 0
    baseline = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(
        color="gray", strokeDash=[3, 3]
    ).encode(y="y:Q")
    layers.append(baseline)

    # Reference: Paris 1.5\u00b0C target
    paris = alt.Chart(pd.DataFrame({"y": [1.5]})).mark_rule(
        color="red", strokeWidth=2
    ).encode(y="y:Q")
    layers.append(paris)

    chart = alt.layer(*layers).properties(height=450).interactive()
    st.altair_chart(chart, use_container_width=True)
    st.caption("Gray dashed = baseline (0\u00b0C). Red line = Paris Agreement 1.5\u00b0C target.")
'''

print(timeseries_code)

### 6.3 Tab 2: Warming Stripes (from Part 1 Step 7.1)

The warming stripes visualization is one of the most effective storytelling charts. Each year is one stripe; the color shift from blue to red makes the warming trend immediately visible.

In [ ]:
stripes_code = '''
with tab2:
    stripes = alt.Chart(annual_filtered).mark_rect().encode(
        x=alt.X("Year:O", axis=alt.Axis(labels=False, ticks=False, title=None)),
        color=alt.Color(
            "Anomaly:Q",
            scale=alt.Scale(scheme="redblue", reverse=True, domain=[-2, 2]),
            legend=alt.Legend(title="Anomaly (\u00b0C)", orient="bottom")
        ),
        tooltip=["Year:O", alt.Tooltip("Anomaly:Q", format=".2f")]
    ).properties(
        height=180,
        title=f"Warming Stripes: {selected_country}"
    )
    st.altair_chart(stripes, use_container_width=True)
    st.caption("Each stripe = one year. Blue \u2192 Red = cooler \u2192 warmer than baseline.")
'''

print(stripes_code)

### 6.4 Tab 3: Decade Comparison (from Part 1 Step 7.2)

The decade bar chart shows how warming has accelerated. This was one of the strongest storytelling visuals in Part 1.

In [ ]:
decade_code = '''
with tab3:
    decade_avg = get_decade_data(annual_filtered)

    bars = alt.Chart(decade_avg).mark_bar().encode(
        x=alt.X("Decade:O", title="Decade"),
        y=alt.Y("Anomaly:Q", title="Average Anomaly (\u00b0C)"),
        color=alt.condition(
            alt.datum.Anomaly > 0,
            alt.value("#e74c3c"),
            alt.value("#3498db")
        ),
        tooltip=["Decade:O", alt.Tooltip("Anomaly:Q", format=".3f")]
    ).properties(height=400)

    st.altair_chart(bars, use_container_width=True)
    st.caption("Red = warmer than baseline. Blue = cooler than baseline.")
'''

print(decade_code)

👉 Checkpoint: Each tab directly reuses a chart type from Part 1. Open your Part 1 notebook and match each tab to its original cell.

<br>

---

## Step 7 · Build the Meaning: Auto-Generated Narrative

This is where the dashboard goes beyond a static report. The narrative updates dynamically based on the user’s selections, turning data into a coherent story.

<br>

### 7.1 Dynamic Story Generator

In [ ]:
story_code = '''
st.markdown("---")
st.header("Data Story")

latest = annual_filtered.iloc[-1]
hottest = annual_filtered.loc[annual_filtered["Anomaly"].idxmax()]
avg = annual_filtered["Anomaly"].mean()

mid = (year_range[0] + year_range[1]) // 2
early_avg = annual_filtered[annual_filtered["Year"] < mid]["Anomaly"].mean()
recent_avg = annual_filtered[annual_filtered["Year"] >= mid]["Anomaly"].mean()
change = recent_avg - early_avg

# The f-string generates a narrative using the Hook-Evidence-Meaning structure
# Hook: latest anomaly value
# Evidence: average, hottest year, period comparison
# Meaning: interpretation of the trend direction
'''

print(story_code)
print("See the complete app.py in Step 9 for the full narrative f-string.")

### 7.2 Paris Agreement Progress

```python
st.subheader("Paris Agreement Tracker")

progress_val = min(latest["Anomaly"] / 1.5, 1.0)
col_a, col_b = st.columns([3, 1])
with col_a:
    st.progress(max(0.0, progress_val))
with col_b:
    st.write(f"**{progress_val * 100:.0f}%** of 1.5\u00b0C")
```

### 7.3 Why This Matters

The narrative section directly implements the **Meaning** layer from Part 1 Step 8. Instead of a fixed paragraph in a notebook, the story now adapts to whatever the user selects. If they switch from South Korea to Brazil, the hook, evidence, and meaning all update automatically.

This is the core advantage of a dashboard over a static report: **the story is alive**.

👉 Try: Change the year range in the sidebar and watch how the story text changes.

<br>

---

## Step 8 · Add a Data Download

Let users export the filtered data, just as you exported outputs in Part 1 Step 9.

```python
st.markdown("---")

csv = annual_filtered.to_csv(index=False)
st.download_button(
    label="Download Filtered Data (CSV)",
    data=csv,
    file_name=f"{selected_country.replace(' ', '_')}_climate.csv",
    mime="text/csv"
)

st.caption("Data: Berkeley Earth Surface Temperature Study | Built with Streamlit")
```

👉 This gives the audience the same export capability you used in Part 1, but through a browser.

<br>

---

## Step 9 · Assemble the Complete `app.py`

You have now built every section of the dashboard. The complete `app.py` combines all the pieces above into one file.

Save the following as `app.py` in your project folder. (A ready-to-use copy is also provided with this tutorial.)

The code is structured to mirror the storytelling framework:

```text
┌─────────────────────────────────────────┐
│  HOOK          Metric cards at top      │
├─────────────────────────────────────────┤
│  EVIDENCE      3 tabbed charts          │
│                 ├─ Time Series          │
│                 ├─ Warming Stripes      │
│                 └─ Decade Bars          │
├─────────────────────────────────────────┤
│  MEANING       Auto-generated narrative │
│                 + Paris Agreement bar   │
├─────────────────────────────────────────┤
│  ACTION        Download data button     │
└─────────────────────────────────────────┘
```

In [ ]:
# The complete app.py is provided as a separate file with this tutorial.
# You can also copy it from the code block in the markdown cell below.
print("See the next cell for the complete app.py source code.")

### Complete `app.py` Source Code

```python
"""
Climate Data Story — Interactive Dashboard
Run locally:  streamlit run app.py
"""

import streamlit as st
import pandas as pd
import altair as alt

# Page Config
st.set_page_config(page_title="Climate Data Story", page_icon="\U0001f30d", layout="wide")

# Data Layer (cached)
@st.cache_data
def load_data():
    df = pd.read_csv("data/ddbb_surface_temperature_countries.csv")
    df = df.dropna(subset=["Anomaly"])
    return df

@st.cache_data
def get_annual_data(df, country):
    country_data = df[df["Country"] == country]
    annual = country_data.groupby("Years").agg(
        {"Anomaly": "mean", "Temperature": "mean"}
    ).reset_index()
    annual.columns = ["Year", "Anomaly", "Temperature"]
    return annual

@st.cache_data
def get_decade_data(annual_df):
    df = annual_df.copy()
    df["Decade"] = (df["Year"] // 10) * 10
    return df.groupby("Decade")["Anomaly"].mean().reset_index()

df = load_data()
countries = sorted(df["Country"].unique())

# Sidebar Controls
st.sidebar.title("\U0001f30d Climate Explorer")
st.sidebar.markdown("Explore temperature trends by country and time period.")
selected_country = st.sidebar.selectbox(
    "Select Country", countries,
    index=countries.index("South Korea") if "South Korea" in countries else 0
)
year_range = st.sidebar.slider("Year Range", 1900, 2020, (1900, 2020))
st.sidebar.markdown("---")
st.sidebar.subheader("Chart Options")
show_trend = st.sidebar.checkbox("Show Trend Line", value=True)
show_ma = st.sidebar.checkbox("Show Moving Average", value=True)

# Prepare Data
annual = get_annual_data(df, selected_country)
annual_filtered = annual[
    (annual["Year"] >= year_range[0]) & (annual["Year"] <= year_range[1])
]

# HOOK — Key Metrics
st.title(f"Climate Story: {selected_country}")
col1, col2, col3, col4 = st.columns(4)
latest = annual_filtered.iloc[-1]
avg_anomaly = annual_filtered["Anomaly"].mean()
hottest = annual_filtered.loc[annual_filtered["Anomaly"].idxmax()]
coldest = annual_filtered.loc[annual_filtered["Anomaly"].idxmin()]
with col1:
    st.metric("Latest Anomaly", f"{latest['Anomaly']:.2f}\u00b0C")
with col2:
    st.metric("Period Average", f"{avg_anomaly:.2f}\u00b0C")
with col3:
    st.metric(f"Hottest ({int(hottest['Year'])})", f"{hottest['Anomaly']:.2f}\u00b0C")
with col4:
    st.metric(f"Coldest ({int(coldest['Year'])})", f"{coldest['Anomaly']:.2f}\u00b0C")
st.markdown("---")

# EVIDENCE — Tabbed Charts
tab1, tab2, tab3 = st.tabs(["Time Series", "Warming Stripes", "Decades"])

with tab1:
    # ... (time series chart code from Step 6.2)
    pass

with tab2:
    # ... (warming stripes code from Step 6.3)
    pass

with tab3:
    # ... (decade bars code from Step 6.4)
    pass

# MEANING — Auto-Generated Narrative
# ... (story generator from Step 7)

# ACTION — Download
# ... (download button from Step 8)
```

The full working version is available as `app.py` alongside this notebook.

👉 Checkpoint: Run `streamlit run app.py` locally. Try selecting different countries and year ranges. Does the story remain coherent?

<br>

---

## Step 10 · Deploy to Streamlit Community Cloud (Free)

Now you will publish the dashboard so anyone can access it through a URL.

<br>

### 10.1 Prepare Your GitHub Repository

1. Create a new repository on [github.com](https://github.com)
2. Push your project files:

```bash
cd data-storytelling-tutorial
git init
git add .
git commit -m "Initial commit: data-storytelling-dashboard"
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/data-storytelling-tutorial.git
git push -u origin main
```

Your repository should contain:

```text
data-storytelling-tutorial/
├── PART1_TUTORIAL.ipynb
├── PART2_TUTORIAL.ipynb
├── app.py
├── requirements.txt
├── data/
│   └── ddbb_surface_temperature_countries.csv
└── README.md
```

<br>

### 10.2 Deploy on Streamlit Community Cloud

1. Go to [share.streamlit.io](https://share.streamlit.io)
2. Sign in with your GitHub account
3. Click **"New app"**
4. Select your repository, branch (`main`), and main file (`app.py`)
5. Click **"Deploy"**

Streamlit will install the packages from `requirements.txt` and launch the app. After a few minutes, you will receive a public URL like:

```
https://YOUR_USERNAME-climate-dashboard-app-xxxxx.streamlit.app
```

<br>

### 10.3 Troubleshooting Deployment

<table>
<tr><td width="240"><b>Problem</b></td><td><b>Solution</b></td></tr>
<tr><td>"File not found" error</td><td>Check that the data path in <code>app.py</code> matches the repository structure: <code>data/ddbb_surface_temperature_countries.csv</code></td></tr>
<tr><td>App crashes on load</td><td>Check that <code>requirements.txt</code> includes all packages (streamlit, pandas, altair, numpy, scipy)</td></tr>
<tr><td>App is slow</td><td>Verify that <code>@st.cache_data</code> is applied to data loading functions</td></tr>
<tr><td>Changes not showing</td><td>Push to GitHub again — Streamlit Cloud auto-redeploys on new commits</td></tr>
</table>

👉 Checkpoint: After deployment, open the URL on your phone. Does the dashboard work on mobile?

<br>

---

## Step 11 · Optional Extensions

Use these exercises to deepen the project after deployment.

<br>

### Exercise 1 · Multi-Country Comparison Tab

Add a fourth tab where users can select multiple countries and compare their trends on one chart. This replicates Part 1 Step 7.4 in an interactive form.

### Exercise 2 · Custom Narrative Controls

Add a form where users choose which statistics to include in the auto-generated story.

### Exercise 3 · Session State for Bookmarking

Use `st.session_state` to remember the user’s last selections across interactions.

In [ ]:
exercise_hint = '''
# Exercise 1 hint: Multi-Country Comparison

with tab4:
    compare_countries = st.multiselect(
        "Select countries to compare",
        countries,
        default=["South Korea", "Japan", "Germany", "Brazil"]
    )

    compare_data = []
    for c in compare_countries:
        c_annual = get_annual_data(df, c)
        c_annual["Country"] = c
        compare_data.append(c_annual)

    if compare_data:
        compare_df = pd.concat(compare_data)
        multi_chart = alt.Chart(compare_df).mark_line().encode(
            x="Year:Q",
            y="Anomaly:Q",
            color="Country:N",
            tooltip=["Country:N", "Year:Q", alt.Tooltip("Anomaly:Q", format=".2f")]
        ).properties(height=400).interactive()
        st.altair_chart(multi_chart, use_container_width=True)
'''

print(exercise_hint)

<br>

---

## Part 2 Summary

### What You Built

<table>
<tr><td width="200"><b>Dashboard Section</b></td><td><b>Story Role</b></td><td><b>Part 1 Origin</b></td></tr>
<tr><td>Metric cards</td><td>Hook</td><td>Step 5 (extremes, statistics)</td></tr>
<tr><td>Time series tab</td><td>Evidence</td><td>Steps 6.3–6.4 (trend chart)</td></tr>
<tr><td>Warming stripes tab</td><td>Evidence</td><td>Step 7.1 (warming stripes)</td></tr>
<tr><td>Decade bars tab</td><td>Evidence</td><td>Step 7.2 (decade bars)</td></tr>
<tr><td>Narrative block</td><td>Meaning</td><td>Step 8 (narrative template)</td></tr>
<tr><td>Paris tracker</td><td>Meaning</td><td>Key concept (Paris Agreement)</td></tr>
<tr><td>Download button</td><td>Action</td><td>Step 9 (export)</td></tr>
</table>

<br>

### Key Takeaways

- A dashboard is a **living story**: Hook, Evidence, and Meaning update automatically
- Streamlit’s **top-to-bottom execution** model means every widget change refreshes the whole narrative
- **Caching** ensures the data layer runs efficiently despite constant reruns
- **Deployment to Streamlit Cloud** is free and requires only a GitHub repository
- The strongest dashboards combine **interactive charts** with **dynamic narrative text**

<br>

---

## Resources

<table>
<tr><td width="160">Streamlit Docs</td><td><a href="https://docs.streamlit.io/">docs.streamlit.io</a></td></tr>
<tr><td>Streamlit Cloud</td><td><a href="https://share.streamlit.io/">share.streamlit.io</a></td></tr>
<tr><td>Streamlit Gallery</td><td><a href="https://streamlit.io/gallery">streamlit.io/gallery</a></td></tr>
<tr><td>Altair Docs</td><td><a href="https://altair-viz.github.io/">altair-viz.github.io</a></td></tr>
<tr><td>Streamlit Cheat Sheet</td><td><a href="https://docs.streamlit.io/library/cheatsheet">docs.streamlit.io/library/cheatsheet</a></td></tr>
</table>

<br>

---

## Next Step

Your climate data story is now live on the web. Share the URL, collect feedback, and iterate. Consider adding more countries, refining the narrative text, or connecting additional datasets to expand the story.